#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, when, count, to_date, length, mean
from pyspark.sql import Window

In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos

def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()
    
    return

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.sales_details")

In [0]:
%skip
info(df) 

#Transformations

## Rename columns' names

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimm

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
%skip
display(df.limit(20))

## Unique Values

In [0]:
%skip
info(df)

display(df.limit(20))

In [0]:
%skip
#Explorando os valores duplicados
id = df.groupby(["order_number", "product_number"]).count()

window_spec = Window.partitionBy("order_number")

id_expandido = id.withColumn(
    "total_produtos_no_pedido", 
    count("product_number").over(window_spec)
).orderBy(col("total_produtos_no_pedido").desc(), col("order_number"))


display(id_expandido.limit(20))


## Missings Values

### Sales Amout, Price

In [0]:
%skip
#Identificando as colunas com Vazios
info(df)

In [0]:
%skip
#Entendendo os Casos
df.show(n=10)
#Filtrando os casos
df.filter(col("sales_amount").isNull()
          | col("price").isNull()).display()



In [0]:
#Tratando os Casos
df_alt = df.withColumn(
    "sales_amount",
    when(
        (col("sales_amount").isNull()) | (col("sales_amount") <= 0),
        col("quantity") * col("price")
    ).otherwise(col("sales_amount"))
    ).withColumn(
        "price",
        when(
            (col("price").isNull()) | (col("price") <= 0),
            col("sales_amount") / col("quantity")
        ).otherwise(col("price"))
    )

#Validando o tratamento
info(df_alt)

## Dates

In [0]:

#Explorando os casos
display(df_alt.limit(5))

#Nomeando as Colunas com Datas
COLUNAS_DATAS = ["order_date", "ship_date", "due_date"]

#Entendndo os casos em cada Coluna

df_alt.withColumn("media de colunas", sum([length(col(colunas)) for colunas in COLUNAS_DATAS])).groupBy("media de colunas").count().display()
print("Há datas fora do padrão")

df_alt.withColumn("media de colunas", sum([length(col(colunas)) for colunas in COLUNAS_DATAS])).filter(col("media de colunas") != 24).display()


In [0]:
df_alt2 = df_alt

for coluna in COLUNAS_DATAS:
    df_alt2 = df_alt2.withColumn(coluna, 
                                when( (col(coluna) == 0) | (length(col(coluna)) != 8), None)
                             .otherwise(to_date(col(coluna).cast("string"), "yyyyMMdd"))
    )

display(df_alt2.limit(5))

#Write into Silver Layer

In [0]:
df_alt2.write.mode("overwrite").saveAsTable("workspace.silver.crm_sales_details")
